# 02 - Baseline OLS (Table 1)

Reproduces the entire-sample safe-haven regression of the thesis (Table 1) on
the free-data snapshot from `01_data`. For each currency the daily **excess
return** is regressed on five risk factors, estimated with **Newey-West (HAC)**
standard errors, `maxlags = 2`:

$$ \text{excess}_t = \alpha + \beta_1 \text{SP500}_t + \beta_2 \text{10Y}_t
   + \beta_3 \text{fxvol}_t + \beta_4 \text{TED}_t + \beta_5 \text{VIX}_t + \varepsilon_t $$

Variable construction follows the thesis: excess return = log FX appreciation +
interest-rate differential; SP500 and VIX as daily log changes; TED = US 3M
interbank − 3M T-bill; fx volatility = 30-day rolling volatility averaged over
the *other* three currencies. Rates enter daily-ized (annual / 360).


## Setup

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

CURRENCIES = ["CHF", "EUR", "GBP", "JPY"]
RATE = {"CHF": "CHF_3M", "EUR": "EUR_3M", "GBP": "GBP_3M", "JPY": "JPY_3M"}

## Load and construct variables

Returns and volatility are built on a **gap-free FX panel** (days with any missing exchange rate are dropped first), matching the thesis's handling of incomplete days.

In [2]:
levels = pd.read_csv(Path("..") / "data" / "levels.csv",
                     index_col="date", parse_dates=True)

# Gap-free FX panel: drop days with any missing exchange rate (holidays), so the
# daily returns and rolling volatility are computed on a continuous series
# (matches the thesis, which omits incomplete days).
fx = levels[CURRENCIES].dropna()
appreciation = np.log(fx).diff()

sp500_d = np.log(levels["SP500"]).diff().rename("SP500")
vix_d = np.log(levels["VIX"]).diff().rename("VIX")

ust10y_d = levels["UST10Y"].diff().rename("UST10Y")
ted_d = ((levels["US_3M"] - levels["TBILL_3M"]) / 360).rename("TED")

# interbank rates are % p.a.; /100/360 gives the daily decimal carry
interest_diff = pd.DataFrame(
    {cur: (levels[RATE[cur]] - levels["US_3M"]) / 100 / 360 for cur in CURRENCIES})
excess = appreciation + interest_diff

## FX volatility

30-day rolling standard deviation of log returns; each currency's regressor is
the average realized volatility of the *other* three (leave-one-out).

In [3]:
realized = np.log(appreciation.rolling(30).std())
fx_vol = pd.DataFrame(
    {cur: realized[[o for o in CURRENCIES if o != cur]].mean(axis=1)
     for cur in CURRENCIES})
fx_vol = fx_vol - fx_vol.mean()   # centre log-vol so it does not offset the intercept

## Per-currency regression (Newey-West, lag 2)

In [4]:
def estimate(cur):
    df = pd.concat([
        excess[cur].rename("excess"),
        sp500_d, ust10y_d, fx_vol[cur].rename("fxvol"), ted_d, vix_d,
    ], axis=1).dropna()
    model = smf.ols("excess ~ SP500 + UST10Y + fxvol + TED + VIX", data=df)
    return model.fit(cov_type="HAC", cov_kwds={"maxlags": 2})

results = {cur: estimate(cur) for cur in CURRENCIES}

In [5]:
# Robustness: the thesis' Newey-West lag of 2 is short for daily data with a
# 30-day rolling regressor; compare t-values against longer bandwidths.
hac = {}
for cur in CURRENCIES:
    df = pd.concat([
        excess[cur].rename("excess"),
        sp500_d, ust10y_d, fx_vol[cur].rename("fxvol"), ted_d, vix_d,
    ], axis=1).dropna()
    auto = int(4 * (len(df) / 100) ** (2 / 9))          # Newey-West rule of thumb
    model = smf.ols("excess ~ SP500 + UST10Y + fxvol + TED + VIX", data=df)
    hac[cur] = pd.DataFrame(
        {f"lag {lag}": model.fit(cov_type="HAC", cov_kwds={"maxlags": lag}).tvalues
         for lag in (2, auto, 30)}).drop("Intercept").round(1)
pd.concat(hac, axis=1)

L = [r"\begin{table}[H]", r"\centering",
     r"\caption{Baseline $t$-statistics under alternative Newey-West bandwidths: "
     r"2 lags (thesis), the Newey-West (1994) rule of thumb, and 30 lags.}",
     r"\label{tab:hac}", r"\begin{tabular}{ll" + "r" * 3 + "}", r"\toprule",
     r"Currency & Regressor & " + " & ".join(hac["CHF"].columns) + r" \\", r"\midrule"]
for cur in CURRENCIES:
    for i, reg in enumerate(hac[cur].index):
        name = cur if i == 0 else ""
        L.append(f"{name} & {reg} & " + " & ".join(
            f"${hac[cur].loc[reg, c]:.1f}$" for c in hac[cur].columns) + r" \\")
    if cur != CURRENCIES[-1]:
        L.append(r"\midrule")
L += [r"\bottomrule", r"\end{tabular}", r"\end{table}"]
(Path("..") / "paper").mkdir(exist_ok=True)
(Path("..") / "paper" / "hac_robustness.tex").write_text("\n".join(L))
print("Wrote ../paper/hac_robustness.tex")

Wrote ../paper/hac_robustness.tex


## Table 1 - reproduced coefficients

Coefficient with t-statistic beneath, per currency.

In [6]:
TERMS = ["Intercept", "SP500", "UST10Y", "fxvol", "TED", "VIX"]

def cell(res, term):
    return f"{res.params[term]:.3f} (t={res.tvalues[term]:.1f})"

table1 = pd.DataFrame(
    {cur: {term: cell(results[cur], term) for term in TERMS} for cur in CURRENCIES})
table1.loc["N"] = {cur: int(results[cur].nobs) for cur in CURRENCIES}
table1.loc["R2"] = {cur: round(results[cur].rsquared, 3) for cur in CURRENCIES}
table1

,CHF,EUR,GBP,JPY
Intercept,0.000 (t=1.2),0.000 (t=0.8),0.000 (t=1.5),-0.000 (t=-1.2)
SP500,0.030 (t=2.5),0.068 (t=5.5),0.085 (t=5.3),-0.009 (t=-0.6)
UST10Y,-0.022 (t=-11.2),-0.013 (t=-7.1),-0.008 (t=-4.1),-0.035 (t=-16.9)
fxvol,0.000 (t=1.1),0.000 (t=0.6),0.000 (t=0.6),0.001 (t=1.9)
TED,-0.066 (t=-0.6),-0.084 (t=-0.7),-0.215 (t=-1.7),0.119 (t=0.9)
VIX,0.004 (t=2.2),0.002 (t=1.2),0.000 (t=0.2),0.009 (t=4.1)
N,5790,5790,5790,5286
R2,0.04,0.023,0.029,0.141


## Comparison with the thesis

Thesis Table 1 coefficients (Bloomberg/Refinitiv data). Signs and significance
patterns should broadly align; magnitudes are **not** directly comparable, because
several regressors are constructed differently here: fxvol enters on a log scale
(tiny coefficients by construction), the 10Y term as a daily *yield change* in
percentage points, and TED from monthly interbank rates. On the 10Y specifically,
the thesis describes it ambiguously as the "interest rate of the 10-year Treasury
future", but its expectations section states it is "seen here as a yield" with a
negative sign expected -- matching its mostly-negative Table 1 estimates. This
reproduction uses the yield (DGS10) explicitly, so the negative signs **agree**
with the thesis for CHF/EUR/JPY; GBP is the only sign difference.

In [7]:
thesis = pd.DataFrame({
    "CHF": {"SP500": -0.021, "UST10Y": -0.504, "fxvol": 0.449, "TED": -0.442, "VIX": 0.003, "Intercept": -0.001},
    "EUR": {"SP500": 0.040, "UST10Y": -0.079, "fxvol": 0.536, "TED": -0.152, "VIX": 0.001, "Intercept": -0.004},
    "GBP": {"SP500": 0.079, "UST10Y": 0.268, "fxvol": 0.510, "TED": -0.446, "VIX": 0.001, "Intercept": -0.005},
    "JPY": {"SP500": -0.067, "UST10Y": -1.348, "fxvol": 0.634, "TED": -0.913, "VIX": 0.009, "Intercept": 0.006},
})

reproduced = pd.DataFrame(
    {cur: {term: results[cur].params[term] for term in TERMS} for cur in CURRENCIES})

comparison = pd.concat({"thesis": thesis, "reproduced": reproduced.round(3)},
                       axis=1).swaplevel(axis=1).sort_index(axis=1)
comparison

CHF               EUR               GBP               JPY  \
          reproduced thesis reproduced thesis reproduced thesis reproduced   
SP500          0.030 -0.021      0.068  0.040      0.085  0.079     -0.009   
UST10Y        -0.022 -0.504     -0.013 -0.079     -0.008  0.268     -0.035   
fxvol          0.000  0.449      0.000  0.536      0.000  0.510      0.001   
TED           -0.066 -0.442     -0.084 -0.152     -0.215 -0.446      0.119   
VIX            0.004  0.003      0.002  0.001      0.000  0.001      0.009   
Intercept      0.000 -0.001      0.000 -0.004      0.000 -0.005     -0.000   

                  
          thesis  
SP500     -0.067  
UST10Y    -1.348  
fxvol      0.634  
TED       -0.913  
VIX        0.009  
Intercept  0.006

## Where does the R² come from?

Re-fit each currency dropping UST10Y, and with UST10Y alone, to show how much of the fit the ten-year yield accounts for. For the JPY almost the entire R² (0.128 of 0.141) comes from the bond-yield term.

In [8]:
# R^2 decomposition: full model vs. dropping UST10Y vs. UST10Y alone.
r2 = {}
for cur in CURRENCIES:
    df = pd.concat([excess[cur].rename("excess"), sp500_d, ust10y_d,
                    fx_vol[cur].rename("fxvol"), ted_d, vix_d], axis=1).dropna()
    full = smf.ols("excess ~ SP500 + UST10Y + fxvol + TED + VIX", data=df).fit()
    no10 = smf.ols("excess ~ SP500 + fxvol + TED + VIX", data=df).fit()
    only = smf.ols("excess ~ UST10Y", data=df).fit()
    r2[cur] = {"R2 full": full.rsquared,
               "R2 without UST10Y": no10.rsquared,
               "R2 UST10Y only": only.rsquared}
pd.DataFrame(r2).T.round(3)

,R2 full,R2 without UST10Y,R2 UST10Y only
CHF,0.040,0.003,0.038
EUR,0.023,0.008,0.008
GBP,0.029,0.024,0.001
JPY,0.141,0.043,0.128


## Full model output (reference)

In [9]:
results["JPY"].summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                 excess   R-squared:                       0.141
Model:                            OLS   Adj. R-squared:                  0.140
Method:                 Least Squares   F-statistic:                     90.36
Date:                Thu, 13 Aug 2026   Prob (F-statistic):           1.68e-91
Time:                        00:29:35   Log-Likelihood:                 19786.
No. Observations:                5286   AIC:                        -3.956e+04
Df Residuals:                    5280   BIC:                        -3.952e+04
Df Model:                           5                                         
Covariance Type:                  HAC                                         
==============================================================================
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     -0.0001      0.000     -1.235      0.217      -0.000     8.6e-05
SP500         -0.0089      0.014     -0.634      0.526      -0.037       0.019
UST10Y        -0.0353      0.002    -16.926      0.000      -0.039      -0.031
fxvol          0.0005      0.000      1.852      0.064   -2.92e-05       0.001
TED            0.1193      0.127      0.941      0.347      -0.129       0.368
VIX            0.0085      0.002      4.088      0.000       0.004       0.013
==============================================================================
Omnibus:                      811.307   Durbin-Watson:                   2.145
Prob(Omnibus):                  0.000   Jarque-Bera (JB):             9689.959
Skew:                           0.334   Prob(JB):                         0.00
Kurtosis:                       9.599   Cond. No.                         912.
==============================================================================

Notes:
[1] Standard Errors are heteroscedasticity and autocorrelation robust (HAC) using 2 lags and without small sample correction
"""

## Persist

In [10]:
out = Path("..") / "data" / "table1_reproduced.csv"
reproduced.round(4).to_csv(out)
print(f"Saved -> {out}")

Saved -> ../data/table1_reproduced.csv


## Export Table 1 to LaTeX

Write the baseline table (coefficient, significance stars, Newey-West standard
error, N, R²) to `../paper/table1.tex`, which the paper includes via
`\input{table1.tex}`. Re-running this notebook regenerates the table in the
paper, the notebook stays the single source of truth.

In [11]:
def _star(p):
    return "^{***}" if p < 0.01 else "^{**}" if p < 0.05 else "^{*}" if p < 0.10 else ""

ROWS = [("SP500", "SP500"), ("UST10Y", "UST10Y"), ("fxvol", "fxvol"),
        ("TED", "TED"), ("VIX", "VIX"), ("Intercept", "Constant")]

lines = [r"\begin{table}[H]", r"\centering",
         r"\caption{Excess-return regression: static OLS estimates. Newey-West (HAC) "
         r"standard errors in parentheses, lag 2. Significance: $^{*}p<0.1$, $^{**}p<0.05$, $^{***}p<0.01$.}",
         r"\label{tab:baseline}",
         r"\begin{tabular}{lrrrr}", r"\toprule",
         " & " + " & ".join(CURRENCIES) + r" \\", r"\midrule"]
for key, label in ROWS:
    lines.append(f"{label} & " + " & ".join(
        f"${results[c].params[key]:.3f}{_star(results[c].pvalues[key])}$" for c in CURRENCIES) + r" \\")
    lines.append(" & " + " & ".join(f"$({results[c].bse[key]:.3f})$" for c in CURRENCIES) + r" \\")
lines += [r"\midrule",
          "Observations & " + " & ".join(str(int(results[c].nobs)) for c in CURRENCIES) + r" \\",
          r"$R^2$ & " + " & ".join(f"{results[c].rsquared:.3f}" for c in CURRENCIES) + r" \\",
          r"\bottomrule", r"\end{tabular}", r"\end{table}"]

out = Path("..") / "paper"
out.mkdir(exist_ok=True)
(out / "table1.tex").write_text("\n".join(lines))
print(f"Wrote {out / 'table1.tex'}")

Wrote ../paper/table1.tex
